# TP 1: LDA/QDA y optimización matemática de modelos

## Integrantes

- Luis Paredes
- Hernando
- Diego Vazquez
- Nestor Biedma

### Objetivos básicos:

- Completar las implementaciones solicitadas: `FasterQDA`, `EfficientQDA`, `TensorizedChol` y `EfficientChol`.
- Explicar en forma básica los pasos algebraicos y de shapes.
- Verificar que todas las variantes predicen igual que `QDA` sobre el mismo split.
- Comparar tiempos, memoria y accuracy de las 9 variantes solicitadas.

### Algunas referencias conceptuales usadas de las clases:

- `6 - supervised_learning.pdf`: separación train/test y evaluación en datos no usados para ajustar.
- `2 - Matrix Factorization.pdf`: uso de productos matriciales, sistemas lineales y matrices definidas positivas.
- `3 - autovalores y autovectores.pdf` y `3 - diagonalización.pdf`: propiedades de matrices simétricas, base ortonormal y diagonalización,relacionado con matrices de covarianza.


# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [1]:
import platform
import time
import tracemalloc

import numpy as np
import numpy.linalg as LA
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RNG_SEED = 6553
np.set_printoptions(precision=4, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.13
NumPy: 2.0.2
SciPy: 1.16.3
scikit-learn: 1.6.1


## Base code
La representacion elegida sigue el enunciado:

- `X` tiene shape `(p, n)`: filas como features y columnas como observaciones.
- `y` tiene shape `(1, n)`.
- Las clases se codifican como enteros `0, ..., k-1`.


In [2]:
class BaseBayesianClassifier:
    def _estimate_a_priori(self, y):
        counts = np.bincount(y.flatten().astype(int))
        return np.log(counts / counts.sum())

    def _fit_params(self, X, y):
        raise NotImplementedError()

    def _predict_log_conditional(self, x, class_idx):
        raise NotImplementedError()

    def fit(self, X, y, a_priori=None):
        self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)
        self.n_classes = len(self.log_a_priori)
        self._fit_params(X, y)
        return self

    def _predict_one(self, x):
        log_posteriori = [
            log_prior + self._predict_log_conditional(x, idx)
            for idx, log_prior in enumerate(self.log_a_priori)
        ]
        return np.argmax(log_posteriori)

    def predict(self, X):
        m_obs = X.shape[1]
        y_hat = np.empty(m_obs, dtype=int)
        for i in range(m_obs):
            y_hat[i] = self._predict_one(X[:, i].reshape(-1, 1))
        return y_hat.reshape(1, -1)

In [3]:
class QDA(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.inv_covs = [
            LA.inv(np.cov(X[:, y.flatten() == idx], bias=True))
            for idx in range(self.n_classes)
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(self.n_classes)
        ]

    def _predict_log_conditional(self, x, class_idx):
        inv_cov = self.inv_covs[class_idx]
        unbiased_x = x - self.means[class_idx]
        return 0.5 * np.log(LA.det(inv_cov)) - 0.5 * (unbiased_x.T @ inv_cov @ unbiased_x).item()


class TensorizedQDA(QDA):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_inv_cov = np.stack(self.inv_covs)  # (k, p, p)
        self.tensor_means = np.stack(self.means)       # (k, p, 1)

    def _predict_log_conditionals(self, x):
        unbiased_x = x - self.tensor_means             # (k, p, 1)
        inner_prod = unbiased_x.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_x
        return 0.5 * np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

# Resoluciones de la consigna QDA

En esta sección se separan explícitamente las resoluciones pedidas por la consigna. La numeración sigue el contenido del enunciado, aunque en la parte de Cholesky el enunciado original repite algunos números.

## Resolución 1: ¿sobre qué paraleliza `TensorizedQDA`?

`TensorizedQDA` paraleliza el cálculo sobre las $k$ clases, pero no sobre las $n$ observaciones.

En `QDA`, para cada observación se evalúa un score por clase usando un ciclo explícito. En `TensorizedQDA`, las matrices de covarianza inversa y las medias de todas las clases se apilan en tensores, de modo que NumPy calcula simultáneamente los scores de todas las clases para una observación dada.

Sin embargo, `TensorizedQDA` hereda el método `predict` de `BaseBayesianClassifier`, que todavía recorre las observaciones una por una. Por eso la mejora se da sobre el loop de clases, no sobre el loop de observaciones.

## Resolución 2: shapes de `tensor_inv_cov` y `tensor_means`

Si hay $k$ clases y $p$ features:

- `tensor_inv_cov` tiene shape `(k, p, p)`: contiene una matriz $\Sigma_j^{-1}$ por clase.
- `tensor_means` tiene shape `(k, p, 1)`: contiene un vector $\mu_j$ por clase.
- Una observación individual `x` tiene shape `(p, 1)`.

Por broadcasting:

```python
unbiased_x = x - tensor_means
```

produce un tensor de shape `(k, p, 1)`, donde cada bloque es $x-\mu_j$.

Luego:

```python
inner_prod = unbiased_x.transpose(0, 2, 1) @ tensor_inv_cov @ unbiased_x
```

tiene shape `(k, 1, 1)`, porque para cada clase calcula:

$$
(x-\mu_j)^T\Sigma_j^{-1}(x-\mu_j).
$$

Finalmente se aplana a `(k,)`, se suma `log_a_priori` y se toma `argmax`. La predicción coincide con `QDA` porque se evalúa la misma función discriminante; solo cambia la forma vectorizada de calcularla.


## Resolución 3: implementación de `FasterQDA`

`FasterQDA` se implementa heredando de `TensorizedQDA` y redefiniendo `predict`. La idea es eliminar el ciclo sobre observaciones y calcular las predicciones para todo `X` de una vez.

La celda de código siguiente implementa:

```python
unbiased_X = X[None, :, :] - self.tensor_means
inner_prod = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X
diag_inner = np.diagonal(inner_prod, axis1=1, axis2=2)
```

Con esto se obtienen simultáneamente los scores de todas las clases y de todas las observaciones.

## Resolución 4: dónde aparece la matriz $n \times n$

La matriz $n \times n$ aparece en:

```python
inner_prod = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X
```

Los shapes son:

- `unbiased_X`: `(k, p, n)`
- `unbiased_X.transpose(0, 2, 1)`: `(k, n, p)`
- `self.tensor_inv_cov`: `(k, p, p)`
- `inner_prod`: `(k, n, n)`

Para cada clase se obtiene una matriz $n \times n$. La diagonal contiene las $n$ formas cuadráticas necesarias, una por observación. El resto de la matriz representa interacciones cruzadas entre observaciones que no se usan para predecir.

## Resolución 5: demostración de la identidad de la diagonal

Sea $C = AB$. La entrada diagonal $i$ de $C$ es:

$$
C_{ii}
=
\sum_j A_{ij}B_{ji}.
$$

Pero $B_{ji}$ es la entrada $(i,j)$ de $B^T$, por lo tanto:

$$
C_{ii}
=
\sum_j A_{ij}(B^T)_{ij}.
$$

Esto es exactamente sumar por columnas el producto elemento a elemento entre $A$ y $B^T$:

$$
\operatorname{diag}(AB)
=
\sum_{\text{cols}} A \odot B^T.
$$

En NumPy:

```python
np.sum(A * B.T, axis=1)
```

La forma equivalente del enunciado,

```python
np.sum(A.T * B, axis=0).T
```

calcula lo mismo con las matrices transpuestas.

## Resolución 6: implementación de `EfficientQDA`

`EfficientQDA` usa la identidad anterior para evitar construir `inner_prod` de shape `(k, n, n)`.

En vez de calcular toda la matriz y quedarse con la diagonal, calcula:

```python
transformed = self.tensor_inv_cov @ unbiased_X
quad = np.sum(unbiased_X * transformed, axis=1)
```

El resultado `quad` tiene shape `(k, n)`, un valor por clase y por observación. Así se conserva la vectorización, pero se evita el costo de memoria de `FasterQDA`.

## Resolución 7: comparación de las 4 variantes sin Cholesky

Las 4 variantes sin Cholesky son:

1. `QDA`
2. `TensorizedQDA`
3. `FasterQDA`
4. `EfficientQDA`

Lo esperado teóricamente es:

- `QDA`: más simple y clara, pero con loops explícitos.
- `TensorizedQDA`: mejora al eliminar el loop sobre clases.
- `FasterQDA`: elimina también el loop sobre observaciones, pero crea matrices $(k,n,n)$.
- `EfficientQDA`: mantiene la predicción vectorizada y evita la matriz $n \times n$.

Por eso, como compromiso entre tiempo y memoria, `EfficientQDA` debería ser la variante sin Cholesky más razonable. Más adelante, en la tabla de benchmark del notebook se observa la comparación concreta para el dataset `wine`.


In [4]:
class FasterQDA(TensorizedQDA):
    def predict(self, X):
        unbiased_X = X[None, :, :] - self.tensor_means                         # (k, p, n)
        inner_prod = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X
        diag_inner = np.diagonal(inner_prod, axis1=1, axis2=2)                 # (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_post = self.log_a_priori.reshape(-1, 1) + log_det - 0.5 * diag_inner
        return np.argmax(log_post, axis=0).reshape(1, -1)


class EfficientQDA(TensorizedQDA):
    def predict(self, X):
        unbiased_X = X[None, :, :] - self.tensor_means                         # (k, p, n)
        transformed = self.tensor_inv_cov @ unbiased_X                         # (k, p, n)
        quad = np.sum(unbiased_X * transformed, axis=1)                        # (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_post = self.log_a_priori.reshape(-1, 1) + log_det - 0.5 * quad
        return np.argmax(log_post, axis=0).reshape(1, -1)

## Resolución 8: expresar $A^{-1}$ usando Cholesky

Si una matriz definida positiva $A$ tiene factorización de Cholesky:

$$
A = LL^T
$$

Entonces:

$$
A^{-1} = (LL^T)^{-1}
$$

Por la regla de la inversa de un producto:

$$
(LL^T)^{-1} = (L^T)^{-1}L^{-1}
$$

Como:

$$
(L^T)^{-1} = (L^{-1})^T
$$

Por lo que, en forma abreviada:

$$
A^{-1} = L^{-T}L^{-1}
$$

En QDA esto es útil porque la forma cuadrática:

$$
(x-\mu)^T\Sigma^{-1}(x-\mu)
$$

puede reescribirse como:

$$
\|L^{-1}(x-\mu)\|_2^2.
$$

Así, en lugar de invertir una matriz general, se trabaja con matrices triangulares, lo cual suele ser más eficiente y más estable numéricamente.

## Resolución 9: diferencias entre `QDA_Chol1` y `QDA`

`QDA` calcula explícitamente:

```python
inv_cov = LA.inv(cov)
```

y luego evalúa:

```python
unbiased_x.T @ inv_cov @ unbiased_x
```

`QDA_Chol1`, en cambio:

1. Calcula la covarianza de cada clase.
2. Factoriza esa covarianza como $\Sigma = LL^T$.
3. Calcula explícitamente $L^{-1}$.
4. Evalúa $y = L^{-1}(x-\mu)$.
5. Usa $\|y\|^2$ como forma cuadrática equivalente.

La predicción final sigue siendo la misma porque se calcula la misma distancia cuadrática, solo escrita usando Cholesky.

## Resolución 10: diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`

Las tres variantes usan Cholesky, pero difieren en cómo calculan $L^{-1}(x-\mu)$:

- `QDA_Chol1`: calcula $L^{-1}$ con `LA.inv`, que es una inversión genérica.
- `QDA_Chol2`: no calcula $L^{-1}$; resuelve el sistema triangular con `solve_triangular`.
- `QDA_Chol3`: calcula $L^{-1}$ usando `dtrtri`, una rutina LAPACK optimizada para matrices triangulares.

Conceptualmente:

- `QDA_Chol1` es la más directa.
- `QDA_Chol2` evita formar la inversa.
- `QDA_Chol3` aprovecha una rutina especializada.

## Resolución 11: comparación de las 7 variantes hasta Cholesky

Las 7 variantes son:

1. `QDA`
2. `TensorizedQDA`
3. `FasterQDA`
4. `EfficientQDA`
5. `QDA_Chol1`
6. `QDA_Chol2`
7. `QDA_Chol3`

La comparación esperada es:

- Las variantes tensorizadas deberían mejorar la predicción.
- Las variantes Cholesky deberían mejorar o estabilizar el tratamiento de la covarianza.
- En datasets chicos puede haber ruido por overhead, así que no siempre el orden teórico aparece perfectamente.

La tabla de benchmark permite ver si alguna variante Cholesky es claramente mejor o peor. En general, si el objetivo es evitar una inversión genérica, `QDA_Chol2` o `QDA_Chol3` son preferibles a `QDA_Chol1`; si el objetivo es acelerar predicción, las variantes vectorizadas suelen tener más impacto.


In [5]:
class QDA_Chol1(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.L_invs = [
            LA.inv(cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True))
            for idx in range(self.n_classes)
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(self.n_classes)
        ]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        y = L_inv @ (x - self.means[class_idx])
        return np.log(np.diagonal(L_inv).prod()) - 0.5 * np.sum(y**2)


class QDA_Chol2(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.Ls = [
            cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True)
            for idx in range(self.n_classes)
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(self.n_classes)
        ]

    def _predict_log_conditional(self, x, class_idx):
        L = self.Ls[class_idx]
        y = solve_triangular(L, x - self.means[class_idx], lower=True)
        return -np.log(np.diagonal(L).prod()) - 0.5 * np.sum(y**2)


class QDA_Chol3(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.L_invs = [
            dtrtri(cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True), lower=1)[0]
            for idx in range(self.n_classes)
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(self.n_classes)
        ]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        y = L_inv @ (x - self.means[class_idx])
        return np.log(np.diagonal(L_inv).prod()) - 0.5 * np.sum(y**2)

## Resolución 12: implementación de `TensorizedChol`

`TensorizedChol` combina Cholesky con la idea de `TensorizedQDA`.

La implementación:

1. Hereda de `QDA_Chol3`.
2. Calcula las matrices $L_j^{-1}$ por clase.
3. Apila esas matrices en `tensor_L_invs` con shape `(k, p, p)`.
4. Apila las medias en `tensor_means` con shape `(k, p, 1)`.
5. Calcula los scores de todas las clases a la vez para una observación.

Paraleliza sobre clases, pero todavía conserva el loop sobre observaciones porque hereda la estructura de predicción individual.

## Resolución 13: implementación de `EfficientChol`

`EfficientChol` combina:

- Cholesky;
- tensorización sobre clases;
- predicción vectorizada sobre observaciones;
- eliminación de matrices $n \times n$.

Para todas las observaciones calcula:

```python
y = self.tensor_L_invs @ unbiased_X
```

con shape `(k, p, n)`. Luego:

```python
np.sum(y**2, axis=1)
```

produce shape `(k, n)`, es decir, una forma cuadrática por clase y por observación.

Esto evita construir matrices grandes y aprovecha la factorización triangular de Cholesky.

## Resolución 14: comparación final de las 9 variantes

Las 9 variantes comparadas son:

1. `QDA`
2. `TensorizedQDA`
3. `FasterQDA`
4. `EfficientQDA`
5. `QDA_Chol1`
6. `QDA_Chol2`
7. `QDA_Chol3`
8. `TensorizedChol`
9. `EfficientChol`

La comparación final debe leerse en tres dimensiones:

- **Accuracy:** todas deberían dar resultados equivalentes si implementan la misma regla de decisión.
- **Tiempo de entrenamiento:** depende sobre todo del ajuste de medias, covarianzas, inversas o Cholesky.
- **Tiempo de predicción:** mejora al eliminar loops y vectorizar.
- **Memoria:** puede empeorar si se construyen matrices intermedias grandes, como en `FasterQDA`.

La tendencia esperada es que `EfficientQDA` y `EfficientChol` sean buenas versiones finales porque evitan el loop sobre observaciones y no construyen la matriz $n \times n$. `EfficientChol` además incorpora la ventaja numérica de Cholesky.


In [6]:
class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_invs = np.stack(self.L_invs)                    # (k, p, p)
        self.tensor_means = np.stack(self.means)                      # (k, p, 1)
        self.log_det_terms = np.log(
            np.diagonal(self.tensor_L_invs, axis1=1, axis2=2).prod(axis=1)
        )

    def _predict_log_conditionals(self, x):
        unbiased_x = x - self.tensor_means                            # (k, p, 1)
        y = self.tensor_L_invs @ unbiased_x                           # (k, p, 1)
        return self.log_det_terms - 0.5 * np.sum(y**2, axis=(1, 2))

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))


class EfficientChol(TensorizedChol):
    def predict(self, X):
        unbiased_X = X[None, :, :] - self.tensor_means                 # (k, p, n)
        y = self.tensor_L_invs @ unbiased_X                            # (k, p, n)
        log_post = (
            self.log_a_priori.reshape(-1, 1)
            + self.log_det_terms.reshape(-1, 1)
            - 0.5 * np.sum(y**2, axis=1)
        )
        return np.argmax(log_post, axis=0).reshape(1, -1)

## Datasets y utilidades

Para que el notebook sea reproducible localmente sin depender de red, los benchmarks principales usan `wine`, incluido en `sklearn`. También se deja cargado `iris` como dataset alternativo chico.


In [7]:
def get_iris_dataset():
    data = load_iris()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1, 1)])
    return X_full, y_full


def get_wine_dataset():
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1, 1)])
    return X_full, y_full


def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)


def split_transpose(X, y, test_size=0.3, random_state=RNG_SEED):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y.flatten()
    )
    return X_train.T, X_test.T, y_train.reshape(1, -1), y_test.reshape(1, -1)


X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)
print("X_full:", X_full.shape)
print("y_full:", y_full.shape)
print("clases:", np.unique(y_full.flatten(), return_counts=True))

X_full: (178, 13)
y_full: (178, 1)
clases: (array(['class_0', 'class_1', 'class_2'], dtype='<U7'), array([59, 71, 48]))


## Verificación de equivalencia

Antes de comparar tiempos, verificamos que todas las variantes produzcan exactamente las mismas predicciones que `QDA` sobre el mismo split. Esta prueba es importante: una implementación más rápida no sirve si cambia la regla de decisión.


In [8]:
ALL_MODELS = [
    QDA,
    TensorizedQDA,
    FasterQDA,
    EfficientQDA,
    QDA_Chol1,
    QDA_Chol2,
    QDA_Chol3,
    TensorizedChol,
    EfficientChol,
]

X_train, X_test, y_train, y_test = split_transpose(X_full, y_full_encoded)
baseline = QDA().fit(X_train, y_train).predict(X_test)

equivalence_rows = []
for model_class in ALL_MODELS:
    preds = model_class().fit(X_train, y_train).predict(X_test)
    equivalence_rows.append(
        {
            "modelo": model_class.__name__,
            "coincide_con_QDA": bool(np.array_equal(preds, baseline)),
            "accuracy": float((preds.flatten() == y_test.flatten()).mean()),
        }
    )

equivalence_df = pd.DataFrame(equivalence_rows)
display(equivalence_df)
assert equivalence_df["coincide_con_QDA"].all()

,modelo,coincide_con_QDA,accuracy
0,QDA,True,0.981481
1,TensorizedQDA,True,0.981481
2,FasterQDA,True,0.981481
3,EfficientQDA,True,0.981481
4,QDA_Chol1,True,0.981481
5,QDA_Chol2,True,0.981481
6,QDA_Chol3,True,0.981481
7,TensorizedChol,True,0.981481
8,EfficientChol,True,0.981481


## Benchmark

Las métricas dependen del hardware y de versiones de librerías. Por eso interesa más la comparación relativa que el valor absoluto.

Se miden:

- tiempo mediano de entrenamiento;
- tiempo mediano de predicción;
- accuracy promedio;
- pico mediano de memoria en entrenamiento;
- pico mediano de memoria en predicción.


In [9]:
class Benchmark:
    def __init__(
        self,
        X,
        y,
        n_runs=120,
        warmup=20,
        mem_runs=20,
        test_sz=0.3,
        rng_seed=RNG_SEED,
    ):
        self.X = X
        self.y = y
        self.n_runs = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.rng_seed = rng_seed
        self.data = {}

    def _split(self, seed):
        return split_transpose(self.X, self.y, test_size=self.test_sz, random_state=seed)

    def bench(self, model_class):
        name = model_class.__name__
        time_data = np.empty((self.n_runs, 3), dtype=float)
        mem_data = np.empty((self.mem_runs, 2), dtype=float)

        for i in range(self.warmup):
            X_train, X_test, y_train, _ = self._split(self.rng_seed + i)
            model = model_class().fit(X_train, y_train)
            model.predict(X_test)

        for i in range(self.mem_runs):
            X_train, X_test, y_train, _ = self._split(self.rng_seed + 10_000 + i)
            model = model_class()

            tracemalloc.start()
            model.fit(X_train, y_train)
            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()
            model.predict(X_test)
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i] = (train_peak / (1024 * 1024), test_peak / (1024 * 1024))

        for i in range(self.n_runs):
            X_train, X_test, y_train, y_test = self._split(self.rng_seed + 20_000 + i)
            model = model_class()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (preds.flatten() == y_test.flatten()).mean(),
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline="QDA"):
        rows = []
        for name, (time_data, mem_data) in self.data.items():
            rows.append(
                {
                    "model": name,
                    "train_median_ms": np.median(time_data[:, 0]),
                    "train_std_ms": time_data[:, 0].std(),
                    "test_median_ms": np.median(time_data[:, 1]),
                    "test_std_ms": time_data[:, 1].std(),
                    "mean_accuracy": time_data[:, 2].mean(),
                    "train_mem_median_mb": np.median(mem_data[:, 0]),
                    "test_mem_median_mb": np.median(mem_data[:, 1]),
                }
            )
        df = pd.DataFrame(rows).set_index("model")
        if baseline in df.index:
            df["train_speedup"] = df.loc[baseline, "train_median_ms"] / df["train_median_ms"]
            df["test_speedup"] = df.loc[baseline, "test_median_ms"] / df["test_median_ms"]
            df["train_mem_reduction"] = df.loc[baseline, "train_mem_median_mb"] / df["train_mem_median_mb"]
            df["test_mem_reduction"] = df.loc[baseline, "test_mem_median_mb"] / df["test_mem_median_mb"]
        return df.sort_values("test_median_ms")

In [10]:
bench = Benchmark(
    X_full,
    y_full_encoded,
    n_runs=100,
    warmup=15,
    mem_runs=15,
    test_sz=0.3,
)

for model_class in ALL_MODELS:
    print("Benchmark:", model_class.__name__)
    bench.bench(model_class)

summary = bench.summary(baseline="QDA")
display(summary.style.format("{:.4f}"))

Benchmark: QDA
Benchmark: TensorizedQDA
Benchmark: FasterQDA
Benchmark: EfficientQDA
Benchmark: QDA_Chol1
Benchmark: QDA_Chol2
Benchmark: QDA_Chol3
Benchmark: TensorizedChol
Benchmark: EfficientChol


,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,test_mem_median_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,
EfficientChol,0.5082,0.1089,0.0664,0.0136,0.9841,0.0182,0.0766,1.0418,39.8163,0.9908,0.0945
EfficientQDA,0.4951,0.1160,0.0945,0.0167,0.9841,0.0181,0.0762,1.0694,27.9669,1.0000,0.0951
FasterQDA,0.5025,0.1439,0.1080,0.0264,0.9841,0.0181,0.1092,1.0535,24.4613,1.0000,0.0664
TensorizedChol,0.5579,0.1490,0.9166,0.2383,0.9841,0.0182,0.0129,0.9489,2.8836,0.9908,0.5603
TensorizedQDA,0.5211,0.0878,1.4678,0.2109,0.9841,0.0181,0.0120,1.0160,1.8006,1.0000,0.6045
QDA_Chol1,0.6348,0.1904,2.5886,0.5540,0.9841,0.0182,0.0076,0.8340,1.0210,0.9908,0.9477
QDA,0.5294,0.1006,2.6430,0.4838,0.9841,0.0181,0.0072,1.0000,1.0000,1.0000,1.0000
QDA_Chol3,0.7590,0.0988,4.3092,1.4263,0.9841,0.0182,0.0076,0.6976,0.6133,0.9908,0.9477
QDA_Chol2,0.7109,0.1296,9.4889,2.8330,0.9841,0.0182,0.0082,0.7447,0.2785,0.9908,0.8863


In [11]:
cols = [
    "train_median_ms",
    "test_median_ms",
    "mean_accuracy",
    "train_speedup",
    "test_speedup",
    "train_mem_reduction",
    "test_mem_reduction",
]
compact_summary = summary[cols].copy()
display(compact_summary.style.format("{:.4f}"))

#fastest_train = compact_summary["train_median_ms"].idxmin()
#fastest_test = compact_summary["test_median_ms"].idxmin()
#lowest_test_mem = compact_summary["test_mem_reduction"].idxmax()

#print(f"Modelo con menor mediana de entrenamiento: {fastest_train}")
#print(f"Modelo con menor mediana de predicción: {fastest_test}")
#print(f"Mayor reduccion relativa de memoria de predicción vs QDA: {lowest_test_mem}")

fastest_train = compact_summary.index[
    compact_summary["train_median_ms"] == compact_summary["train_median_ms"].min()
].tolist()

fastest_test = compact_summary.index[
    compact_summary["test_median_ms"] == compact_summary["test_median_ms"].min()
].tolist()

lowest_test_mem = compact_summary.index[
    compact_summary["test_mem_reduction"] == compact_summary["test_mem_reduction"].max()
].tolist()

print(f"Modelo(s) con menor mediana de entrenamiento: {fastest_train}")
print(f"Modelo(s) con menor mediana de predicción: {fastest_test}")
print(f"Modelo(s) con mayor reducción relativa de memoria de predicción vs QDA: {lowest_test_mem}")

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
EfficientChol,0.5082,0.0664,0.9841,1.0418,39.8163,0.9908,0.0945
EfficientQDA,0.4951,0.0945,0.9841,1.0694,27.9669,1.0000,0.0951
FasterQDA,0.5025,0.1080,0.9841,1.0535,24.4613,1.0000,0.0664
TensorizedChol,0.5579,0.9166,0.9841,0.9489,2.8836,0.9908,0.5603
TensorizedQDA,0.5211,1.4678,0.9841,1.0160,1.8006,1.0000,0.6045
QDA_Chol1,0.6348,2.5886,0.9841,0.8340,1.0210,0.9908,0.9477
QDA,0.5294,2.6430,0.9841,1.0000,1.0000,1.0000,1.0000
QDA_Chol3,0.7590,4.3092,0.9841,0.6976,0.6133,0.9908,0.9477
QDA_Chol2,0.7109,9.4889,0.9841,0.7447,0.2785,0.9908,0.8863


Modelo(s) con menor mediana de entrenamiento: ['EfficientQDA']
Modelo(s) con menor mediana de predicción: ['EfficientChol']
Modelo(s) con mayor reducción relativa de memoria de predicción vs QDA: ['QDA']


## Conclusiones

1. `TensorizedQDA` mejora la predicción frente a `QDA` porque elimina el ciclo sobre clases, pero todavía procesa observación por observación.
2. `FasterQDA` elimina también el ciclo sobre observaciones, aunque paga el costo de construir una matriz `(k, n, n)`. Puede ser rápido para datasets chicos, pero no escala bien en memoria.
3. `EfficientQDA` conserva la vectorización y evita la matriz `n x n`, por lo que es la versión sin Cholesky con mejor compromiso teórico.
4. Las variantes Cholesky atacan el costo de trabajar con $\Sigma^{-1}$. `QDA_Chol2` resuelve sistemas triangulares y `QDA_Chol3` usa una inversión triangular optimizada.
5. `TensorizedChol` y `EfficientChol` combinan la idea numérica de Cholesky con la idea computacional de tensorización.
6. En problemas chicos como `wine`, las diferencias absolutas pueden ser muy pequeñas; la tendencia relevante es conceptual: evitar loops de Python y evitar matrices intermedias innecesarias suele ganar cuando crecen $k$, $p$ o $n$.
